## 1. Load data

In [ ]:
# Load a deterministic subset of Open-Orca/OpenOrca (for 0.5B SFT)
# Pin a specific commit for full reproducibility (shown on HF commit history page)
REVISION = "e9c87b4" # 2025-02-19
from datasets import load_dataset, DatasetDict

SUBSET_ROWS = 1000000
train_ds = load_dataset("Open-Orca/OpenOrca",
                        split=f"train[:{SUBSET_ROWS}]",
                        revision=REVISION,)
dataset = DatasetDict({"train": train_ds})

README.md: 0.00B [00:00, ?B/s]

1M-GPT4-Augmented.parquet:   0%|          | 0.00/1.01G [00:00<?, ?B/s]

3_5M-GPT3_5-Augmented.parquet:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

## 2. Data cleaning

- formatting issues
- quality issues
- consistency issues

In [ ]:
import re
import numpy as np
from collections import defaultdict
from datasets import DatasetDict
from tqdm import tqdm
import hashlib

def clean_openorca_dataset(dataset, sample_size=1000000):
    """
    clean OpenOrca dataset, formatting, quality, consistency issues.

    Args:
        dataset: OpenOrca dataset
        sample_size: number of data rows

    Returns:
        cleaned_dataset
    """

    train_dataset = dataset["train"]

    print(f"colunm names: {train_dataset.column_names}")

    cleaned_data = []
    format_issues = 0
    quality_issues = 0
    duplicates_removed = 0
    content_hashes = set()

    for i, example in tqdm(enumerate(train_dataset), total=min(len(train_dataset), sample_size)):
        if i >= sample_size:
            break

        try:
            # conversation retrival
            if 'question' in example and 'response' in example:
                question = str(example['question']).strip()
                response = str(example['response']).strip()
                system_prompt = str(example.get('system_prompt', '')).strip()

                # build conversation format
                conversation = []
                if system_prompt:
                    conversation.append({"role": "system", "content": system_prompt})
                conversation.append({"role": "user", "content": question})
                conversation.append({"role": "assistant", "content": response})

            elif 'conversations' in example:
                conversation = example['conversations']
                if not isinstance(conversation, list):
                    format_issues += 1
                    continue
            else:
                format_issues += 1
                continue

            # 1. format
            if not check_format_issues(conversation):
                format_issues += 1
                continue

            # 2. quality
            if not check_quality_issues(conversation):
                quality_issues += 1
                continue

            # 3. consistency
            full_text = extract_conversation_text(conversation)
            text_hash = hashlib.md5(full_text.encode()).hexdigest()

            # duplicates
            if text_hash in content_hashes:
                duplicates_removed += 1
                continue
            content_hashes.add(text_hash)

            cleaned_data.append({
                'id': example.get('id', f'cleaned_{i}'),
                'conversations': conversation,
                'original_index': i
            })

        except Exception as e:
            format_issues += 1
            continue

    print(f"  - original: {min(len(train_dataset), sample_size)}")
    print(f"  - format removes: {format_issues}")
    print(f"  - quality removes: {quality_issues}")
    print(f"  - consistency removes: {duplicates_removed}")
    print(f"  - remaining: {len(cleaned_data)}")

    # semantic consistency
    if len(cleaned_data) > 0:
        cleaned_data = check_semantic_consistency(cleaned_data)

    cleaned_dataset = DatasetDict({
        "train": train_dataset.select([ex['original_index'] for ex in cleaned_data])
    })

    return cleaned_dataset

def check_format_issues(conversation):
    """
    check format issues

    1. non-standard roles
    2. missing fields
    3. mismatched dialogue turns
    """

    # standard roles
    standard_roles = {'system', 'user', 'assistant', 'human'}

    if not conversation or not isinstance(conversation, list):
        return False

    for turn in conversation:
        if not isinstance(turn, dict):
            return False

        if 'role' not in turn or 'content' not in turn:
            return False

        # check non-standard roles
        role = turn['role'].lower().strip()
        if role not in standard_roles:
            if role in ['instruction', 'prompt', 'input']:
                turn['role'] = 'user'
            elif role in ['output', 'answer', 'reply']:
                turn['role'] = 'assistant'
            else:
                return False

        # check empty fields
        content = str(turn['content']).strip()
        if not content or len(content) == 0:
            return False

    # check dialogue turns
    for i, turn in enumerate(conversation):
        role = turn['role']
        if i == 0 and role == 'system':
            continue
        elif i == 0 and role != 'user':
            return False
        elif i > 0:
            prev_role = conversation[i-1]['role']
            if role == prev_role:
                return False

    return True

def check_quality_issues(conversation):
    """
    check quality issues

    1. empty/garbage text
    2. repetition
    3. templated spam
    4. extremely short/extremely long anomalous samples
    """

    for turn in conversation:
        content = str(turn['content']).strip()

        # 1. empty
        if not content or len(content) == 0:
            return False

        # 2. garbage text
        special_char_ratio = count_special_characters(content) / max(len(content), 1)
        if special_char_ratio > 0.3:
            return False

        # 3. repetition
        if has_character_repetition(content, threshold=0.07):
            return False

        # 4. templated spam
        if is_templated_spam(content):
            return False

        # 5. length
        words = content.split()

        # extremely short
        meaningful_words = [w for w in words if len(w) > 1]
        if len(meaningful_words) < 3:
            return False

        # extremely long
        if len(words) > 5000:
            return False

        if contains_quality_issues(content):
            return False

    return True

def count_special_characters(text):
    """count non-alnum and punctuation ratio"""
    normal_punctuation = set('.,!?;:\'"()- ')
    special_chars = sum(1 for c in text if not c.isalnum() and c not in normal_punctuation)
    return special_chars

def has_character_repetition(text, threshold=0.3):
    """character repetition"""
    if len(text) < 10:
        return False

    # #long sequence
    # for i in range(len(text) - 4):
    #     if text[i:i+2] == text[i+2:i+4] == text[i+4:i+6]:
    #         return True

    # overall repetition
    unique_chars = len(set(text))
    if unique_chars / len(text) < threshold:
        return True

    return False

def is_templated_spam(text):
    """templated spam check"""
    spam_patterns = [
        r'http[s]?://\S+',
        r'@\w+',
        r'#\w+',
        r'[\*_\-]{3,}',
        r'(?:click|visit|buy|subscribe|follow)[\s\S]{1,20}(?:here|link|now)',
        r'password|login|sign\s*in|username',
    ]

    text_lower = text.lower()
    for pattern in spam_patterns:
        if re.search(pattern, text_lower):
            return True

    if len(text) > 20 and sum(1 for c in text if c.isupper()) / len(text) > 0.5:
        return True

    return False

def contains_quality_issues(text):
    issues = [
        '[empty]', '[deleted]', '[removed]', 'undefined',
        'test', 'example', 'placeholder', 'lorem ipsum',
        '...', '??', '!!!', '???'
    ]

    text_lower = text.lower()
    for issue in issues:
        if issue in text_lower:
            return True

    return False

def extract_conversation_text(conversation):
    texts = []
    for turn in conversation:
        content = str(turn['content']).strip()
        content = ' '.join(content.split())
        texts.append(content)
    return ' '.join(texts)

def check_semantic_consistency(cleaned_data, similarity_threshold=0.9):
    """
    semantic consistency

    """
    if len(cleaned_data) < 2:
        return cleaned_data

    prompts_dict = defaultdict(list)

    for item in cleaned_data:
        conversation = item['conversations']
        user_prompts = [turn['content'] for turn in conversation if turn['role'] in ['user', 'human']]
        if user_prompts:
            prompt = user_prompts[0]
            normalized_prompt = normalize_text(prompt)
            prompts_dict[normalized_prompt].append(item)

    final_data = []
    duplicates_found = 0

    for prompt, items in prompts_dict.items():
        if len(items) == 1:
            final_data.append(items[0])
        else:
            best_item = max(items, key=lambda x: len(
                ''.join([turn['content'] for turn in x['conversations']
                        if turn['role'] == 'assistant'])
            ))
            final_data.append(best_item)
            duplicates_found += (len(items) - 1)

    print(f"  - consistency removes: {duplicates_found}")
    return final_data

def normalize_text(text):
    """text normalization"""
    text = text.lower()
    text = ' '.join(text.split())
    text = re.sub(r'[^\w\s]', '', text)
    words = sorted(set(text.split()))
    return ' '.join(words)



In [ ]:
if __name__ == "__main__":

    cleaned_dataset = clean_openorca_dataset(dataset, sample_size=SUBSET_ROWS)

    cleaned_dataset.save_to_disk("/content/cleaned_openorca")

    cleaned_dataset["train"].to_json("/content/cleaned_openorca.json")

colunm names: ['id', 'system_prompt', 'question', 'response']


100%|██████████| 1000000/1000000 [03:45<00:00, 4424.84it/s]


  - original: 1000000
  - format removes: 1
  - quality removes: 728665
  - consistency removes: 838
  - remaining: 270496
  - consistency removes: 30156


Saving the dataset (0/1 shards):   0%|          | 0/240340 [00:00<?, ? examples/s]

Creating json from Arrow format:   0%|          | 0/241 [00:00<?, ?ba/s]

## 3. Fine tuning

In [ ]:
%cd /content/
%rm -rf LLaMA-Factory
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
%cd LLaMA-Factory
%ls
!pip install -e .[torch,bitsandbytes]

/content
Cloning into 'LLaMA-Factory'...
remote: Enumerating objects: 614, done.
remote: Counting objects: 100% (614/614), done.
remote: Compressing objects: 100% (455/455), done.
remote: Total 614 (delta 150), reused 378 (delta 101), pack-reused 0 (from 0)
Receiving objects: 100% (614/614), 5.24 MiB | 29.47 MiB/s, done.
Resolving deltas: 100% (150/150), done.
/content/LLaMA-Factory
assets/       docker/    LICENSE      pyproject.toml  requirements/  tests/
CITATION.cff  docs/      Makefile     README.md       scripts/       tests_v1/
data/         examples/  MANIFEST.in  README_zh.md    src/
Obtaining file:///content/LLaMA-Factory
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import torch
try:
  assert torch.cuda.is_available() is True
except AssertionError:
  print("Please set up a GPU before using LLaMA Factory")

In [ ]:
import json

%cd /content/LLaMA-Factory/

/content/LLaMA-Factory


In [ ]:
import os

dataset_info_path = "data/dataset_info.json"

new_dataset_entry = {
  "open_orca": {
    "file_name": "/content/cleaned_openorca.json",
    "columns": {
      "prompt": "question",
      "response": "response",
      "system": "system_prompt"
    }
  }
}

with open(dataset_info_path, 'r') as f:
    dataset_info = json.load(f)

dataset_info.update(new_dataset_entry)

with open(dataset_info_path, 'w') as f:
    json.dump(dataset_info, f, indent=2)

In [ ]:
torch.cuda.empty_cache()

### Train

- **Key Hyperparameters** to tune

In [ ]:
!CUDA_VISIBLE_DEVICES=0 llamafactory-cli train \
  --model_name_or_path "Qwen/Qwen2.5-0.5B" \
  --stage sft \
  --do_train \
  --dataset open_orca \
  --template qwen \
  --finetuning_type full \
  --output_dir /content/outputs/qwen_openorca_FFT \
  --overwrite_cache \
  --cutoff_len 1024 \
  --per_device_train_batch_size 2 \
  --gradient_accumulation_steps 4 \
  --lr_scheduler_type cosine \
  --warmup_ratio 0.1 \
  --logging_steps 1000 \
  --save_steps 30000 \
  --learning_rate 1e-5 \
  --num_train_epochs 2 \
  --gradient_checkpointing \
  --fp16 \
  --report_to tensorboard

2026-02-11 10:08:22.921289: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-11 10:08:23.308493: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770804503.447090    3012 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770804503.489548    3012 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770804503.829457    3012 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

## Evaluate

In [3]:
!pip -q install -U "lm_eval[hf]" tiktoken sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 9.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 118.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 18.0 MB/s eta 0:00:00


### Base model

In [1]:
import os, json, time
import torch

SFT_MODEL_DIR = "Qwen/Qwen2.5-0.5B"
OUT_DIR = "/content/lm_eval_results/base"
os.makedirs(OUT_DIR, exist_ok=True)
MODEL_ARGS = f"pretrained={SFT_MODEL_DIR},trust_remote_code=True,dtype=float16"
DEVICE = "cuda:0"
BATCH_SIZE = 4

In [ ]:
!python -m lm_eval \
  --model hf \
  --model_args "{MODEL_ARGS}" \
  --tasks mmlu \
  --num_fewshot 5 \
  --device {DEVICE} \
  --batch_size {BATCH_SIZE} \
  --apply_chat_template \
  --output_path "{OUT_DIR}/mmlu_5shot.json"

2026-02-12:01:37:56 INFO     [config.evaluate_config:301] Using default fewshot_as_multiturn=True.
2026-02-12:01:38:01 INFO     [_cli.run:376] Selected Tasks: ['mmlu']
2026-02-12:01:38:02 INFO     [evaluator:211] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-02-12:01:38:02 INFO     [evaluator:236] Initializing hf model, with arguments: {'pretrained': 'Qwen/Qwen2.5-0.5B', 'trust_remote_code': True, 'dtype': 'float16'}
2026-02-12:01:38:05 INFO     [models.huggingface:161] Using device 'cuda:0'
2026-02-12:01:38:06 INFO     [models.huggingface:423] Model parallel was set to False, max memory was not set, and device map was set to {'': 'cuda:0'}
2026-02-12 01:38:06.912469: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variab

In [4]:
# Cell — ARC-E (0-shot)
!python -m lm_eval \
  --model hf \
  --model_args "{MODEL_ARGS}" \
  --tasks arc_easy \
  --num_fewshot 0 \
  --device {DEVICE} \
  --batch_size {BATCH_SIZE} \
  --apply_chat_template \
  --output_path "{OUT_DIR}/arc_easy_0shot.json"

2026-02-12:02:25:01 INFO     [config.evaluate_config:301] Using default fewshot_as_multiturn=True.
2026-02-12:02:25:08 INFO     [_cli.run:376] Selected Tasks: ['arc_easy']
2026-02-12:02:25:10 INFO     [evaluator:211] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-02-12:02:25:10 INFO     [evaluator:236] Initializing hf model, with arguments: {'pretrained': 'Qwen/Qwen2.5-0.5B', 'trust_remote_code': True, 'dtype': 'float16'}
2026-02-12:02:25:19 INFO     [models.huggingface:161] Using device 'cuda:0'
config.json: 100% 681/681 [00:00<00:00, 5.50MB/s]
tokenizer_config.json: 7.23kB [00:00, 32.8MB/s]
vocab.json: 2.78MB [00:00, 115MB/s]
merges.txt: 1.67MB [00:00, 136MB/s]
tokenizer.json: 7.03MB [00:00, 180MB/s]
2026-02-12:02:25:20 INFO     [models.huggingface:423] Model parallel was set to False, max memory was not set, and device map was set to {'': 'cuda:0'}
2026-02-12 02:25:22.111387: I tensorflow/core/util

In [5]:
# Cell — ARC-Challenge (25-shot)
!python -m lm_eval \
  --model hf \
  --model_args "{MODEL_ARGS}" \
  --tasks arc_challenge \
  --num_fewshot 25 \
  --device {DEVICE} \
  --batch_size {BATCH_SIZE} \
  --apply_chat_template \
  --output_path "{OUT_DIR}/arc_challenge_25shot.json"

2026-02-12:02:27:22 INFO     [config.evaluate_config:301] Using default fewshot_as_multiturn=True.
2026-02-12:02:27:27 INFO     [_cli.run:376] Selected Tasks: ['arc_challenge']
2026-02-12:02:27:28 INFO     [evaluator:211] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-02-12:02:27:28 INFO     [evaluator:236] Initializing hf model, with arguments: {'pretrained': 'Qwen/Qwen2.5-0.5B', 'trust_remote_code': True, 'dtype': 'float16'}
2026-02-12:02:27:32 INFO     [models.huggingface:161] Using device 'cuda:0'
2026-02-12:02:27:32 INFO     [models.huggingface:423] Model parallel was set to False, max memory was not set, and device map was set to {'': 'cuda:0'}
2026-02-12 02:27:33.346396: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environme

In [6]:
# Cell — HellaSwag (10-shot)
!python -m lm_eval \
  --model hf \
  --model_args "{MODEL_ARGS}" \
  --tasks hellaswag \
  --num_fewshot 10 \
  --device {DEVICE} \
  --batch_size {BATCH_SIZE} \
  --apply_chat_template \
  --output_path "{OUT_DIR}/hellaswag_10shot.json"

2026-02-12:02:32:19 INFO     [config.evaluate_config:301] Using default fewshot_as_multiturn=True.
2026-02-12:02:32:26 INFO     [_cli.run:376] Selected Tasks: ['hellaswag']
2026-02-12:02:32:27 INFO     [evaluator:211] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-02-12:02:32:27 INFO     [evaluator:236] Initializing hf model, with arguments: {'pretrained': 'Qwen/Qwen2.5-0.5B', 'trust_remote_code': True, 'dtype': 'float16'}
2026-02-12:02:32:30 INFO     [models.huggingface:161] Using device 'cuda:0'
2026-02-12:02:32:31 INFO     [models.huggingface:423] Model parallel was set to False, max memory was not set, and device map was set to {'': 'cuda:0'}
2026-02-12 02:32:31.647534: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment v

In [7]:
# Cell — WinoGrande (5-shot)
!python -m lm_eval \
  --model hf \
  --model_args "{MODEL_ARGS}" \
  --tasks winogrande \
  --num_fewshot 5 \
  --device {DEVICE} \
  --batch_size {BATCH_SIZE} \
  --apply_chat_template \
  --output_path "{OUT_DIR}/winograde_5shot.json"

2026-02-12:03:03:05 INFO     [config.evaluate_config:301] Using default fewshot_as_multiturn=True.
2026-02-12:03:03:10 INFO     [_cli.run:376] Selected Tasks: ['winogrande']
2026-02-12:03:03:12 INFO     [evaluator:211] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-02-12:03:03:12 INFO     [evaluator:236] Initializing hf model, with arguments: {'pretrained': 'Qwen/Qwen2.5-0.5B', 'trust_remote_code': True, 'dtype': 'float16'}
2026-02-12:03:03:15 INFO     [models.huggingface:161] Using device 'cuda:0'
2026-02-12:03:03:15 INFO     [models.huggingface:423] Model parallel was set to False, max memory was not set, and device map was set to {'': 'cuda:0'}
2026-02-12 03:03:16.453242: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment 

In [8]:
# Cell — TruthfulQA_mc2 (0-shot)
!python -m lm_eval \
  --model hf \
  --model_args "{MODEL_ARGS}" \
  --tasks truthfulqa_mc2 \
  --num_fewshot 0 \
  --device {DEVICE} \
  --batch_size {BATCH_SIZE} \
  --apply_chat_template \
  --output_path "{OUT_DIR}/truthfulqa_mc2_0shot.json"

2026-02-12:03:04:01 INFO     [config.evaluate_config:301] Using default fewshot_as_multiturn=True.
2026-02-12:03:04:07 INFO     [_cli.run:376] Selected Tasks: ['truthfulqa_mc2']
2026-02-12:03:04:09 INFO     [evaluator:211] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-02-12:03:04:09 INFO     [evaluator:236] Initializing hf model, with arguments: {'pretrained': 'Qwen/Qwen2.5-0.5B', 'trust_remote_code': True, 'dtype': 'float16'}
2026-02-12:03:04:12 INFO     [models.huggingface:161] Using device 'cuda:0'
2026-02-12:03:04:12 INFO     [models.huggingface:423] Model parallel was set to False, max memory was not set, and device map was set to {'': 'cuda:0'}
2026-02-12 03:04:13.234849: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environm

In [9]:
# Cell — PIQA (0-shot)
!python -m lm_eval \
  --model hf \
  --model_args "{MODEL_ARGS}" \
  --tasks piqa \
  --num_fewshot 0 \
  --device {DEVICE} \
  --batch_size {BATCH_SIZE} \
  --apply_chat_template \
  --output_path "{OUT_DIR}/piqa_0shot.json"

2026-02-12:03:05:28 INFO     [config.evaluate_config:301] Using default fewshot_as_multiturn=True.
2026-02-12:03:05:34 INFO     [_cli.run:376] Selected Tasks: ['piqa']
2026-02-12:03:05:35 INFO     [evaluator:211] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-02-12:03:05:35 INFO     [evaluator:236] Initializing hf model, with arguments: {'pretrained': 'Qwen/Qwen2.5-0.5B', 'trust_remote_code': True, 'dtype': 'float16'}
2026-02-12:03:05:38 INFO     [models.huggingface:161] Using device 'cuda:0'
2026-02-12:03:05:39 INFO     [models.huggingface:423] Model parallel was set to False, max memory was not set, and device map was set to {'': 'cuda:0'}
2026-02-12 03:05:40.227656: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variab

In [10]:
# Cell — BoolQ (0-shot)
!python -m lm_eval \
  --model hf \
  --model_args "{MODEL_ARGS}" \
  --tasks boolq \
  --num_fewshot 0 \
  --device {DEVICE} \
  --batch_size {BATCH_SIZE} \
  --apply_chat_template \
  --output_path "{OUT_DIR}/boolq_0shot.json"

2026-02-12:03:06:29 INFO     [config.evaluate_config:301] Using default fewshot_as_multiturn=True.
2026-02-12:03:06:34 INFO     [_cli.run:376] Selected Tasks: ['boolq']
2026-02-12:03:06:36 INFO     [evaluator:211] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-02-12:03:06:36 INFO     [evaluator:236] Initializing hf model, with arguments: {'pretrained': 'Qwen/Qwen2.5-0.5B', 'trust_remote_code': True, 'dtype': 'float16'}
2026-02-12:03:06:39 INFO     [models.huggingface:161] Using device 'cuda:0'
2026-02-12:03:06:40 INFO     [models.huggingface:423] Model parallel was set to False, max memory was not set, and device map was set to {'': 'cuda:0'}
2026-02-12 03:06:41.136328: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment varia

### SFT model

In [ ]:
import os, json, time
import torch

SFT_MODEL_DIR = "/content/outputs/qwen_openorca_FFT"
OUT_DIR = "/content/lm_eval_results/"
os.makedirs(OUT_DIR, exist_ok=True)
MODEL_ARGS = f"pretrained={SFT_MODEL_DIR},trust_remote_code=True,dtype=float16"
DEVICE = "cuda:0"
BATCH_SIZE = 4

#### MMLU (5-shot)

In [ ]:
!python -m lm_eval \
  --model hf \
  --model_args "{MODEL_ARGS}" \
  --tasks mmlu \
  --num_fewshot 5 \
  --device {DEVICE} \
  --batch_size {BATCH_SIZE} \
  --apply_chat_template \
  --output_path "{OUT_DIR}/mmlu_5shot.json"

2026-02-12:00:25:41 INFO     [config.evaluate_config:301] Using default fewshot_as_multiturn=True.
2026-02-12:00:25:46 INFO     [_cli.run:376] Selected Tasks: ['mmlu']
2026-02-12:00:25:48 INFO     [evaluator:211] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-02-12:00:25:48 INFO     [evaluator:236] Initializing hf model, with arguments: {'pretrained': '/content/outputs/qwen_openorca_FFT', 'trust_remote_code': True, 'dtype': 'float16'}
2026-02-12:00:25:51 INFO     [models.huggingface:161] Using device 'cuda:0'
The tokenizer you are loading from '/content/outputs/qwen_openorca_FFT' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
2026-02-12:00:25:52 INFO     [models.huggingfa

#### ARC-E (0-shot)

In [ ]:
# Cell — ARC-E (0-shot)
!python -m lm_eval \
  --model hf \
  --model_args "{MODEL_ARGS}" \
  --tasks arc_easy \
  --num_fewshot 0 \
  --device {DEVICE} \
  --batch_size {BATCH_SIZE} \
  --apply_chat_template \
  --output_path "{OUT_DIR}/arc_easy_0shot.json"

2026-02-12:00:41:06 INFO     [config.evaluate_config:301] Using default fewshot_as_multiturn=True.
2026-02-12:00:41:11 INFO     [_cli.run:376] Selected Tasks: ['arc_easy']
2026-02-12:00:41:12 INFO     [evaluator:211] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-02-12:00:41:12 INFO     [evaluator:236] Initializing hf model, with arguments: {'pretrained': '/content/outputs/qwen_openorca_FFT', 'trust_remote_code': True, 'dtype': 'float16'}
2026-02-12:00:41:15 INFO     [models.huggingface:161] Using device 'cuda:0'
The tokenizer you are loading from '/content/outputs/qwen_openorca_FFT' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
2026-02-12:00:41:16 INFO     [models.huggi

#### ARC-Challenge (25-shot)

In [ ]:
# Cell — ARC-Challenge (25-shot)
!python -m lm_eval \
  --model hf \
  --model_args "{MODEL_ARGS}" \
  --tasks arc_challenge \
  --num_fewshot 25 \
  --device {DEVICE} \
  --batch_size {BATCH_SIZE} \
  --apply_chat_template \
  --output_path "{OUT_DIR}/arc_challenge_25shot.json"

2026-02-12:00:43:29 INFO     [config.evaluate_config:301] Using default fewshot_as_multiturn=True.
2026-02-12:00:43:36 INFO     [_cli.run:376] Selected Tasks: ['arc_challenge']
2026-02-12:00:43:38 INFO     [evaluator:211] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-02-12:00:43:38 INFO     [evaluator:236] Initializing hf model, with arguments: {'pretrained': '/content/outputs/qwen_openorca_FFT', 'trust_remote_code': True, 'dtype': 'float16'}
2026-02-12:00:43:41 INFO     [models.huggingface:161] Using device 'cuda:0'
The tokenizer you are loading from '/content/outputs/qwen_openorca_FFT' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
2026-02-12:00:43:41 INFO     [models.

#### HellaSwag (10-shot)

In [ ]:
# Cell — HellaSwag (10-shot)
!python -m lm_eval \
  --model hf \
  --model_args "{MODEL_ARGS}" \
  --tasks hellaswag \
  --num_fewshot 10 \
  --device {DEVICE} \
  --batch_size {BATCH_SIZE} \
  --apply_chat_template \
  --output_path "{OUT_DIR}/hellaswag_10shot.json"

2026-02-12:00:49:04 INFO     [config.evaluate_config:301] Using default fewshot_as_multiturn=True.
2026-02-12:00:49:11 INFO     [_cli.run:376] Selected Tasks: ['hellaswag']
2026-02-12:00:49:13 INFO     [evaluator:211] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-02-12:00:49:13 INFO     [evaluator:236] Initializing hf model, with arguments: {'pretrained': '/content/outputs/qwen_openorca_FFT', 'trust_remote_code': True, 'dtype': 'float16'}
2026-02-12:00:49:16 INFO     [models.huggingface:161] Using device 'cuda:0'
The tokenizer you are loading from '/content/outputs/qwen_openorca_FFT' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
2026-02-12:00:49:17 INFO     [models.hugg

#### WinoGrande (5-shot)

In [ ]:
# Cell — WinoGrande (5-shot)
!python -m lm_eval \
  --model hf \
  --model_args "{MODEL_ARGS}" \
  --tasks winogrande \
  --num_fewshot 5 \
  --device {DEVICE} \
  --batch_size {BATCH_SIZE} \
  --apply_chat_template \
  --output_path "{OUT_DIR}/winograde_5shot.json"

2026-02-12:01:20:16 INFO     [config.evaluate_config:301] Using default fewshot_as_multiturn=True.
2026-02-12:01:20:21 INFO     [_cli.run:376] Selected Tasks: ['winogrande']
2026-02-12:01:20:22 INFO     [evaluator:211] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-02-12:01:20:22 INFO     [evaluator:236] Initializing hf model, with arguments: {'pretrained': '/content/outputs/qwen_openorca_FFT', 'trust_remote_code': True, 'dtype': 'float16'}
2026-02-12:01:20:25 INFO     [models.huggingface:161] Using device 'cuda:0'
The tokenizer you are loading from '/content/outputs/qwen_openorca_FFT' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
2026-02-12:01:20:26 INFO     [models.hug

#### TruthfulQA_mc2 (0-shot)

In [ ]:
# Cell — TruthfulQA_mc2 (0-shot)
!python -m lm_eval \
  --model hf \
  --model_args "{MODEL_ARGS}" \
  --tasks truthfulqa_mc2 \
  --num_fewshot 0 \
  --device {DEVICE} \
  --batch_size {BATCH_SIZE} \
  --apply_chat_template \
  --output_path "{OUT_DIR}/truthfulqa_mc2_0shot.json"

2026-02-12:01:21:22 INFO     [config.evaluate_config:301] Using default fewshot_as_multiturn=True.
2026-02-12:01:21:28 INFO     [_cli.run:376] Selected Tasks: ['truthfulqa_mc2']
2026-02-12:01:21:30 INFO     [evaluator:211] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-02-12:01:21:30 INFO     [evaluator:236] Initializing hf model, with arguments: {'pretrained': '/content/outputs/qwen_openorca_FFT', 'trust_remote_code': True, 'dtype': 'float16'}
2026-02-12:01:21:33 INFO     [models.huggingface:161] Using device 'cuda:0'
The tokenizer you are loading from '/content/outputs/qwen_openorca_FFT' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
2026-02-12:01:21:34 INFO     [models

#### PIQA (0-shot)

In [ ]:
# Cell — PIQA (0-shot)
!python -m lm_eval \
  --model hf \
  --model_args "{MODEL_ARGS}" \
  --tasks piqa \
  --num_fewshot 0 \
  --device {DEVICE} \
  --batch_size {BATCH_SIZE} \
  --apply_chat_template \
  --output_path "{OUT_DIR}/piqa_0shot.json"

2026-02-12:01:23:09 INFO     [config.evaluate_config:301] Using default fewshot_as_multiturn=True.
2026-02-12:01:23:15 INFO     [_cli.run:376] Selected Tasks: ['piqa']
2026-02-12:01:23:17 INFO     [evaluator:211] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-02-12:01:23:17 INFO     [evaluator:236] Initializing hf model, with arguments: {'pretrained': '/content/outputs/qwen_openorca_FFT', 'trust_remote_code': True, 'dtype': 'float16'}
2026-02-12:01:23:21 INFO     [models.huggingface:161] Using device 'cuda:0'
The tokenizer you are loading from '/content/outputs/qwen_openorca_FFT' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
2026-02-12:01:23:21 INFO     [models.huggingfa

#### BoolQ (0-shot)

In [ ]:
# Cell — BoolQ (0-shot)
!python -m lm_eval \
  --model hf \
  --model_args "{MODEL_ARGS}" \
  --tasks boolq \
  --num_fewshot 0 \
  --device {DEVICE} \
  --batch_size {BATCH_SIZE} \
  --apply_chat_template \
  --output_path "{OUT_DIR}/boolq_0shot.json"

2026-02-12:01:24:22 INFO     [config.evaluate_config:301] Using default fewshot_as_multiturn=True.
2026-02-12:01:24:27 INFO     [_cli.run:376] Selected Tasks: ['boolq']
2026-02-12:01:24:29 INFO     [evaluator:211] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-02-12:01:24:29 INFO     [evaluator:236] Initializing hf model, with arguments: {'pretrained': '/content/outputs/qwen_openorca_FFT', 'trust_remote_code': True, 'dtype': 'float16'}
2026-02-12:01:24:33 INFO     [models.huggingface:161] Using device 'cuda:0'
The tokenizer you are loading from '/content/outputs/qwen_openorca_FFT' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
2026-02-12:01:24:33 INFO     [models.huggingf